In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import torch

In [ ]:
def generate_synthetic_data(m=8, avg_vars_per_group=4, T=1000, F=10.0):
    """
    生成符合论文设定的人工合成数据
    返回:
    - X: 三维数组，形状为 (m, p_i, T)，其中 p_i 是第 i 组的变量数
    - Y_true: 真实的组级别变量，形状为 (m, T)
    - p_list: 每个组的变量数列表
    """
    total_vars_needed = m * avg_vars_per_group
    p_list = []
    remaining = total_vars_needed
    for i in range(m-1):
        p_i = np.random.randint(2, 2*avg_vars_per_group)
        p_i = min(p_i, remaining - 2*(m-i-1))  
        p_i = max(p_i, 2)
        if p_i % 2 == 1:
            p_i = p_i -1
        p_list.append(p_i)
        remaining -= p_i
    
    p_list.append(max(remaining, 2))
    
    p_list[-1] = total_vars_needed - sum(p_list[:-1])
    
    def lorenz96(t, y, F):
        m = len(y)
        dydt = np.zeros(m)
        
        for i in range(m):
            dydt[i] = (y[(i+1) % m] - y[(i-2) % m]) * y[(i-1) % m] - y[i] + F
        
        return dydt

    np.random.seed(42)  
    y0 = np.random.randn(m) * 0.1  
    
    t_span = (0, 50)  # 在长时间尺度上积分
    t_eval = np.linspace(t_span[0], t_span[1], T*10)  # 高分辨率采样
    
    sol = solve_ivp(
        lorenz96,
        t_span,
        y0,
        args=(F,),
        t_eval=t_eval,
        method='RK45',
        rtol=1e-8,
        atol=1e-10
    )
    
    Y_true = sol.y[:, ::10]  # 降采样
    Y_true = Y_true[:, :T]  # 确保长度为 T
    
    # 标准化每个时间序列
    Y_true = (Y_true - Y_true.mean(axis=1, keepdims=True)) / (Y_true.std(axis=1, keepdims=True) + 1e-8)
    
    
    # 3. 生成观测变量 X
    X = []
    
    for i in range(m):
        p_i = p_list[i]
        
        # 生成权重矩阵 B_i (p_i × 1)，部分随机噪声
        B_i = np.random.randn(p_i, 1) * 0.5 + 1.0
        B_i = B_i / np.linalg.norm(B_i)  # 归一化
        
        # X_i = B_i * Y_i + 小噪声 
        X_i = np.zeros((p_i, T)) 
        for k in range(p_i): 
            X_i[k, :] = B_i[k, 0] * Y_true[i, :] + np.random.randn(T) * 0.05
        
        # 标准化每个观测变量
        for k in range(p_i):
            X_i[k, :] = (X_i[k, :] - X_i[k, :].mean()) / (X_i[k, :].std() + 1e-8)
        
        X.append(X_i)
    
    
    return X, Y_true, p_list


def visualize_data(X, Y_true, p_list):
    """可视化生成的数据"""
    m = len(X)
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    # 1. 显示组级别变量
    ax = axes[0]
    for i in range(min(m, 3)):  # 只显示前3个组
        ax.plot(Y_true[i, :100], label=f'Y_{i+1}')
    ax.set_title('组级别变量 (前100个时间点)')
    ax.set_xlabel('时间')
    ax.set_ylabel('值')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. 显示第一个组的观测变量
    ax = axes[1]
    for k in range(min(p_list[0], 4)):  # 只显示前4个变量
        ax.plot(X[0][k, :100], label=f'X_1,{k+1}')
    ax.set_title('第1组观测变量 (前100个时间点)')
    ax.set_xlabel('时间')
    ax.set_ylabel('值')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 显示第二个组的观测变量
    ax = axes[2]
    for k in range(min(p_list[1], 4)):  # 只显示前4个变量
        ax.plot(X[1][k, :100], label=f'X_2,{k+1}')
    ax.set_title('第2组观测变量 (前100个时间点)')
    ax.set_xlabel('时间')
    ax.set_ylabel('值')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. 显示组内相关性
    ax = axes[3]
    correlations = []
    for i in range(m):
        # 计算每个组内变量之间的平均相关性
        if p_list[i] > 1:
            corr_matrix = np.corrcoef(X[i])
            # 取非对角线元素的平均值
            mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
            avg_corr = corr_matrix[mask].mean()
            correlations.append(avg_corr)
    
    ax.bar(range(len(correlations)), correlations)
    ax.set_title('每个组内的平均相关性')
    ax.set_xlabel('组索引')
    ax.set_ylabel('平均相关系数')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 5. 显示组级别变量的自相关
    ax = axes[4]
    for i in range(min(m, 3)):
        autocorr = np.correlate(Y_true[i, :], Y_true[i, :], mode='full')
        autocorr = autocorr[autocorr.size//2:]
        autocorr = autocorr[:50] / autocorr[0]  # 归一化
        ax.plot(autocorr, label=f'Y_{i+1}')
    ax.set_title('组级别变量自相关')
    ax.set_xlabel('滞后')
    ax.set_ylabel('自相关系数')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. 显示时间序列的功率谱
    ax = axes[5]
    for i in range(min(m, 3)):
        fft_vals = np.abs(np.fft.rfft(Y_true[i, :]))
        freqs = np.fft.rfftfreq(len(Y_true[i, :]))
        ax.plot(freqs[:50], fft_vals[:50], label=f'Y_{i+1}')
    ax.set_title('功率谱密度')
    ax.set_xlabel('频率')
    ax.set_ylabel('幅度')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 7. 显示原始组级别变量的散点图
    ax = axes[6]
    ax.scatter(Y_true[0, :], Y_true[1, :], alpha=0.5, s=1)
    ax.set_title('Y_1 vs Y_2 散点图')
    ax.set_xlabel('Y_1')
    ax.set_ylabel('Y_2')
    ax.grid(True, alpha=0.3)
    
    # 8. 显示观测变量的散点图
    ax = axes[7]
    ax.scatter(X[0][0, :], X[0][1, :], alpha=0.5, s=1)
    ax.set_title('X_{1,1} vs X_{1,2} 散点图')
    ax.set_xlabel('X_{1,1}')
    ax.set_ylabel('X_{1,2}')
    ax.grid(True, alpha=0.3)
    
    # 9. 显示数据统计信息
    ax = axes[8]
    ax.axis('off')
    
    stats_text = f"""
    数据统计信息:
    
    组数: {m}
    总观测变量: {sum(p_list)}
    时间序列长度: {Y_true.shape[1]}
    
    每组变量数:
    """
    for i, p_i in enumerate(p_list):
        stats_text += f"\n  组 {i+1}: {p_i} 个变量"
    
    stats_text += f"""
    
    均值统计:
    Y 均值范围: [{Y_true.mean(axis=1).min():.3f}, {Y_true.mean(axis=1).max():.3f}]
    Y 标准差范围: [{Y_true.std(axis=1).min():.3f}, {Y_true.std(axis=1).max():.3f}]
    """
    
    ax.text(0.1, 0.9, stats_text, transform=ax.transAxes, 
            fontsize=9, verticalalignment='top')
    
    plt.tight_layout()
    plt.show()


def save_data(X, Y_true, p_list, filename='synthetic_data.npz'):
    """保存生成的数据"""
    # 将X转换为列表的列表，以便保存
    X_flat = []
    for i in range(len(X)):
        X_flat.append(X[i])
    
    np.savez_compressed(
        filename,
        X=X_flat,  # 列表的列表
        Y_true=Y_true,
        p_list=np.array(p_list),
        m=len(X)
    )
    print(f"数据已保存到 {filename}")



X, Y_true, p_list = generate_synthetic_data(
    m=8, 
    avg_vars_per_group=4, 
    T=1000,
    F=10.0
)

print("\n" + "="*50)
print("数据形状:")
for i in range(len(X)):
    print(f"  组 {i+1}: {X[i].shape} (变量数 × 时间长度)")

# 可视化数据
# print("\n生成可视化图表...")
# visualize_data(X, Y_true, p_list)


# 保存数据
# save_data(X, Y_true, p_list, 'synthetic_timeseries_data.npz')

# 验证数据质量
print("\n" + "="*50)
print("数据质量验证:")

# 1. 检查组内相关性
# print("\n1. 组内相关性检查:")
# for i in range(min(3, len(X))):  # 只检查前3组
#     if p_list[i] > 1:
#         corr_matrix = np.corrcoef(X[i])
#         mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
#         avg_corr = corr_matrix[mask].mean()
#         print(f"   组 {i+1}: 平均组内相关性 = {avg_corr:.3f}")

# 2. 检查组间相关性
# print("\n2. 组间相关性检查 (组级别变量):")
# for i in range(min(3, len(Y_true))):
#     for j in range(i+1, min(4, len(Y_true))):
#         corr = np.corrcoef(Y_true[i, :], Y_true[j, :])[0, 1]
#         print(f"   Y_{i+1} 和 Y_{j+1}: 相关系数 = {corr:.3f}")

# 3. 检查时间序列特性
# print("\n3. 时间序列特性:")
# for i in range(min(3, len(Y_true))):
#     # 自相关衰减
#     autocorr = np.correlate(Y_true[i, :], Y_true[i, :], mode='full')
#     autocorr = autocorr[autocorr.size//2:]
#     lag10_corr = autocorr[10] / autocorr[0]
#     print(f"   Y_{i+1}: 滞后10自相关 = {lag10_corr:.3f}")

# print("\n数据生成完成！")


数据形状:
  组 1: (6, 1000) (变量数 × 时间长度)
  组 2: (4, 1000) (变量数 × 时间长度)
  组 3: (4, 1000) (变量数 × 时间长度)
  组 4: (4, 1000) (变量数 × 时间长度)
  组 5: (2, 1000) (变量数 × 时间长度)
  组 6: (2, 1000) (变量数 × 时间长度)
  组 7: (2, 1000) (变量数 × 时间长度)
  组 8: (8, 1000) (变量数 × 时间长度)

数据质量验证:


In [3]:
X_new = np.concatenate(X, axis=0).T

In [4]:
p_list

[6, 4, 4, 4, 2, 2, 2, 8]

In [5]:
X_new.shape

(1000, 32)

In [6]:
np.savez('./loc_data_kuramoto/generated_data.npz',
        data=X_new,
        group=np.array(p_list))

In [8]:
pd.DataFrame(X_new).to_csv('./syn_data/all_data.csv', header=None, index=None)
pd.DataFrame(X_new[:-1]).to_csv('./syn_data/train_input.csv', header=None, index=None)
pd.DataFrame(X_new[1:]).to_csv('./syn_data/train_target.csv', header=None, index=None)
pd.DataFrame(X_new[:-1]).to_csv('./syn_data/test_input.csv', header=None, index=None)
pd.DataFrame(X_new[1:]).to_csv('./syn_data/test_target.csv', header=None, index=None)
pd.DataFrame(np.array(p_list)).to_csv('./syn_data/group.csv', header=None, index=None)